In [ ]:
1. On crée le fichier qui permet de bloc-encoder notre matrice C de collision pour qu'elle et on sort le circuit quantique correspondant.

In [3]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from fable import fable

def block_encoding(A, eps=1e-6):
    """Block-encoding FABLE de A, avec vérification : le coin N x N doit valoir A / (alpha*N)."""
    N = A.shape[0]
    circ, alpha = fable(A, eps)
    U = Operator(circ).data                  # matrice exacte du circuit
    err = np.linalg.norm(alpha * N * U[:N, :N] - A) / np.linalg.norm(A)
    return circ, alpha, U, err

tau, omega = 0.8, 1/0.8
w = np.array([2/3, 1/6, 1/6])
C = (1 - omega)*np.eye(3) + omega*np.outer(w, np.ones(3))
C4 = np.zeros((4, 4)); C4[:3, :3] = C          # taille 2^2 exigée par FABLE

circC, alpha, U, err = block_encoding(C4)      # <- le block-encoding de C
print("facteur alpha*N =", alpha*4, " erreur =", err)

# un pas complet : FABLE(C) sur la vitesse, puis streaming direct
n = 3; N = 2**n
def increment(m):
    qc = QuantumCircuit(m)
    for k in reversed(range(1, m)): qc.mcx(list(range(k)), k)
    qc.x(0); return qc.to_gate()

pos, vel, anc = list(range(n)), [n, n+1], [n+2, n+3, n+4]
qc = QuantumCircuit(n + 5)
qc.append(circC.to_gate(), vel + anc)                                # C (block-encodée)
qc.append(increment(n).control(2, ctrl_state=0b01), vel + pos)       # S : bloc P
qc.append(increment(n).inverse().control(2, ctrl_state=0b10), vel + pos)  # S : bloc P^T

# vérification : le coin vaut L / (alpha*4)
P = np.roll(np.eye(N), 1, axis=0); I, Z = np.eye(N), np.zeros((N, N))
S = np.block([[I,Z,Z,Z],[Z,P,Z,Z],[Z,Z,P.T,Z],[Z,Z,Z,I]])
L = S @ np.kron(C4, I)
print("coin = L/4 ?", np.allclose(alpha*4*Operator(qc).data[:4*N, :4*N], L))

facteur alpha*N = 4.0  erreur = 7.359239970340052e-16
coin = L/4 ? True


In [4]:
from qiskit.quantum_info import Statevector

# 1. le fluide : une bosse en x = 2, à l'équilibre
x = np.arange(N)
rho = np.exp(-(x - 2)**2 / 2)
f = np.outer(w, np.ones(1)) * rho                      # tableau (3, N)

# 2. encodage : f -> amplitudes (ancillas à 0)
v = np.zeros(4*N); v[:3*N] = f.ravel()
psi = v / np.linalg.norm(v)
etat = np.zeros(2**(n + 5)); etat[:4*N] = psi

# 3. un pas quantique, puis post-sélection : on garde la partie "ancillas = 0"
sortie = Statevector(etat).evolve(qc).data
garde = np.real(sortie[:4*N])
p = garde @ garde                                       # probabilité que la post-sélection réussisse

# 4. décodage : amplitudes -> tableau f, remis à l'échelle par la masse (conservée)
f_quantique = (garde[:3*N] / garde[:3*N].sum() * f.sum()).reshape(3, N)

# 5. comparaison avec le pas classique
fs = C @ f
f_classique = np.array([fs[0], np.roll(fs[1], 1), np.roll(fs[2], -1)])
print("proba de succès :", round(p, 3))
print("rho quantique   :", f_quantique.sum(0).round(3))
print("rho classique   :", f_classique.sum(0).round(3))
print("écart max       :", np.abs(f_quantique - f_classique).max())

proba de succès : 0.062
rho quantique   : [0.191 0.594 0.869 0.594 0.193 0.03  0.002 0.023]
rho classique   : [0.191 0.594 0.869 0.594 0.193 0.03  0.002 0.023]
écart max       : 9.992007221626409e-16
